kernel : Python (Pixi)

#### Main Dataset

In [1]:
%load_ext autoreload
%autoreload 2
import os
import scanpy as sc
from pipeline.utils.env import find_env_dir
from pipeline.utils import plot

SERIES = "macnair"
clustered_h5ad_dir = find_env_dir("CLUSTERED_H5AD")
final_h5ad_dir = find_env_dir("FINAL_H5AD")
pre_h5ad_dir = find_env_dir("PRE_H5AD")

macnair_adata = sc.read_h5ad(os.path.join(clustered_h5ad_dir, SERIES + ".h5ad"))

In [ ]:
cell_marker_genes = {
    "OPC": ["PDGFRA", "MEGF11", "HAS2", "FERMT1", "CACNG5", "COL9A1", "MYT1", ],
    "COP": ["BCAN", "GPR17", "TCF7L2", "BCAS1", ],
    "Oligodendrocyte": ["MOG", "MAG", "CNP", "MOBP", "PPP1R14A", "ERMN", "GPR37", "KLK6", "TF", "UGT8", "MYRF", ],
    "Neuron": ["STMN2", "CACNG3", "RBFOX3", "SNAP25", "ENO2", "SYT1", 
               "CCK", "GABRA1", "GRIN1", "GRIN2B", "MYT1L", "SYN1", "TMEM130", "CAMK2A", ],
    "Excitatory": ["SLC17A7", "SLC17A6", "NEUROD2", "DRD1", "SATB2", "TBR1", ],
    "Inhibitory": ["GAD1", "GAD2", "DLX1", "DLX2", "DLX5", "DLX6", "SLC32A1", ],
    "Microglia": ["P2RY12", "TMEM119","SLC2A5", "SELPLG", "MROCKI", "CX3CR1", "CSF1R", "ITGAM", "ITGAX", "TREM2", "HLA-DRA", "APOC1", ],
    "Tcell": ["CCL5", "CD3D", "CD3E", "CD8A", "NKG7", "TRAC", "CD28" ],
    "Bcell": ["CD79A", "BANK1", "MS4A1", "XBP1", "SDC1", "MZB1", "TNFRSF17", ], 
    "Endothelial": ["CLDN5", "PECAM1", "CD34", "FLT1", "VWF", "CDH5", "CLEC14A", "ERG", "ITM2A", "TM4SF1", ],
    "Pericyte": ["PDGFRB", "RGS5", "CD248", "KCNJ8", "NOTCH3", ],
    "Astrocyte": ["GFAP", "ALDH1L1", "MLC1", "GJA1", "SLC14A1", "AGT", "BMPR1B", "ID4", "RFX4", "SOX9", "AQP4", ],
    "Ependymal": ["DNAH11", "FOXJ1", "DYNLRB2", "CIMAP3", ],
    "VLMC": ["COL1A2", "COL3A1", "DCN", ],
}
plot.plot_dotplot(macnair_adata, SERIES, cell_marker_genes, group = "cell", project="ANNOTATION")

In [ ]:
leiden_to_celltype = {
    "0": "ExcitatoryNeuron",
    "1": "ExcitatoryNeuron",
    "2": "InhibitoryNeuron",
    "3": "Oligodendrocyte",
    "4": "ExcitatoryNeuron",
    "5": "ExcitatoryNeuron",
    "6": "ExcitatoryNeuron",
    "7": "ExcitatoryNeuron",
    "8": "InhibitoryNeuron",
    "9": "Oligodendrocyte",
    "10": "Astrocyte",
    "11": "Astrocyte",
    "12": "InhibitoryNeuron",
    "13": "ExcitatoryNeuron",
    "14": "ExcitatoryNeuron",
    "15": "ExcitatoryNeuron",
    "16": "Astrocyte",
    "17": "ExcitatoryNeuron",
    "18": "Oligodendrocyte",
    "19": "OPC",
    "20": "ExcitatoryNeuron",
    "21": "ExcitatoryNeuron",
    "22": "UnknownNeuron",
    "23": "ExcitatoryNeuron",
    "24": "InhibitoryNeuron",
    "25": "ExcitatoryNeuron",
    "26": "Oligodendrocyte",
    "27": "ExcitatoryNeuron",
    "28": "ExcitatoryNeuron",
    "29": "Oligodendrocyte",
    "30": "ExcitatoryNeuron",
    "31": "ExcitatoryNeuron",
    "32": "ExcitatoryNeuron",
    "33": "InhibitoryNeuron",
    "34": "Astrocyte",
    "35": "Oligodendrocyte",
    "36": "Oligodendrocyte",
    "37": "Oligodendrocyte",
    "38": "Oligodendrocyte",
    "39": "Oligodendrocyte",
    "40": "Astrocyte",
    "41": "ExcitatoryNeuron",
    "42": "Oligodendrocyte",
    "43": "Oligodendrocyte",
    "44": "Oligodendrocyte",
    "45": "Oligodendrocyte",
    "46": "Oligodendrocyte",
    "47": "Oligodendrocyte",
    "48": "ExcitatoryNeuron",
    "49": "InhibitoryNeuron",
    "50": "OPC",
    "51": "Oligodendrocyte",
    "52": "Tcell",
    "53": "Bcell",
    "54": "Endothelial",
    "55": "Astrocyte",
    "56": "Astrocyte",
    "57": "VLMC",
    "58": "InhibitoryNeuron",
    "59": "Pericyte",
    "60": "Astrocyte",
    "61": "Microglia",
    "62": "Microglia",
    "63": "Microglia", 
    "64": "Ependymal",
    "65": "Microglia",
    "66": "Microglia",
    "67": "Microglia",
    "68": "COP",
}

leiden_str = macnair_adata.obs["leiden"].astype(str)
mapped = leiden_str.map(leiden_to_celltype)

missing = sorted(leiden_str[mapped.isna()].unique())
if missing:
    raise ValueError(
        f"Missing leiden value in leiden_to_celltype : {missing}\n"
    )
macnair_adata.obs["cell"] = mapped
macnair_adata = macnair_adata[macnair_adata.obs["cell"] != "Doublet"].copy()

In [15]:
macnair_adata.obs["celltype"] = macnair_adata.obs["cell"].copy() #type: ignore
plot.plot_umap(macnair_adata, SERIES, has_celltype=True)

In [ ]:
macnair_adata.write_h5ad(os.path.join(final_h5ad_dir, SERIES + "_final.h5ad"))

oligo = macnair_adata[
    (macnair_adata.obs["cell"] == "OPC") |
    (macnair_adata.obs["cell"] == "COP") | 
    (macnair_adata.obs["cell"] == "Oligodendrocyte")
]
oligo.write_h5ad(os.path.join(pre_h5ad_dir, SERIES + "_oligo_raw.h5ad"))

microglia = macnair_adata[
    (macnair_adata.obs["cell"] == "Microglia")
]
microglia.write_h5ad(os.path.join(pre_h5ad_dir, SERIES + "_microglia_raw.h5ad"))

astrocyte = macnair_adata[
    (macnair_adata.obs["cell"] == "Astrocyte")
]
astrocyte.write_h5ad(os.path.join(pre_h5ad_dir, SERIES + "_astrocyte_raw.h5ad"))

/tmp/ipykernel_2789143/1537763218.py:10: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  oligo.write_h5ad(os.path.join(pre_h5ad_dir, SERIES + "_oligo_raw.h5ad"))


#### Validation Dataset

In [4]:
import os
import scanpy as sc
from pipeline.utils.env import find_env_dir
from pipeline.utils import plot

SERIES = "macnair_validation"
clustered_h5ad_dir = find_env_dir("CLUSTERED_H5AD")
final_h5ad_dir = find_env_dir("FINAL_H5AD")

macnair_validation_adata = sc.read_h5ad(os.path.join(clustered_h5ad_dir, SERIES + ".h5ad"))

In [ ]:
cell_marker_genes = {
    "OPC": ["PDGFRA", "MEGF11", "HAS2", "FERMT1", "CACNG5", "COL9A1", "MYT1", ],
    "COP": ["BCAN", "GPR17", "TCF7L2", "BCAS1", ],
    "Oligodendrocyte": ["MOG", "MAG", "CNP", "MOBP", "PPP1R14A", "ERMN", "GPR37", "KLK6", "TF", "UGT8", "MYRF", ],
    "Neuron": ["STMN2", "CACNG3", "RBFOX3", "SNAP25", "ENO2", "SYT1", 
               "CCK", "GABRA1", "GRIN1", "GRIN2B", "MYT1L", "SYN1", "TMEM130", "CAMK2A", ],
    "Excitatory": ["SLC17A7", "SLC17A6", "NEUROD2", "DRD1", "SATB2", "TBR1", ],
    "Inhibitory": ["GAD1", "GAD2", "DLX1", "DLX2", "DLX5", "DLX6", "SLC32A1", ],
    "Microglia": ["P2RY12", "TMEM119","SLC2A5", "SELPLG", "MROCKI", "CX3CR1", "CSF1R", "ITGAM", "ITGAX", "TREM2", "HLA-DRA", "APOC1", ],
    "Tcell": ["CCL5", "CD3D", "CD3E", "CD8A", "NKG7", "TRAC", "CD28" ],
    "Bcell": ["CD79A", "BANK1", "MS4A1", "XBP1", "SDC1", "MZB1", "TNFRSF17", ], 
    "Endothelial": ["CLDN5", "PECAM1", "CD34", "FLT1", "VWF", "CDH5", "CLEC14A", "ERG", "ITM2A", "TM4SF1", ],
    "Pericyte": ["PDGFRB", "RGS5", "CD248", "KCNJ8", "NOTCH3", ],
    "Astrocyte": ["GFAP", "ALDH1L1", "MLC1", "GJA1", "SLC14A1", "AGT", "BMPR1B", "ID4", "RFX4", "SOX9", "AQP4", ],
    "Ependymal": ["DNAH11", "FOXJ1", "DYNLRB2", "CIMAP3", ],
    "VLMC": ["COL1A2", "COL3A1", "DCN", ],
}
plot.plot_dotplot(macnair_validation_adata, SERIES, cell_marker_genes, group = "cell", project="ANNOTATION")

In [5]:
leiden_to_celltype = {
    "0": "Doublet",
    "1": "Oligodendrocyte",
    "2": "Oligodendrocyte",
    "3": "Oligodendrocyte",
    "4": "Oligodendrocyte",
    "5": "Oligodendrocyte",
    "6": "Microglia",
    "7": "InhibitoryNeuron",
    "8": "Astrocyte",
    "9": "Oligodendrocyte",
    "10": "Oligodendrocyte",
    "11": "Oligodendrocyte",
    "12": "Oligodendrocyte",
    "13": "Pericyte",
    "14": "Oligodendrocyte",
    "15": "Oligodendrocyte",
    "16": "Oligodendrocyte",
    "17": "Oligodendrocyte",
    "18": "Oligodendrocyte",
    "19": "Microglia",
    "20": "Astrocyte",
    "21": "Oligodendrocyte",
    "22": "ExcitatoryNeuron",
    "23": "Oligodendrocyte",
    "24": "Oligodendrocyte",
    "25": "Oligodendrocyte",
    "26": "Astrocyte",
    "27": "ExcitatoryNeuron",
    "28": "OPC",
    "29": "Endothelial",
    "30": "Tcell",
    "31": "Astrocyte",
    "32": "VLMC",
    "33": "ExcitatoryNeuron",
    "34": "InhibitoryNeuron",
    "35": "Oligodendrocyte",
    "36": "ExcitatoryNeuron",
    "37": "Oligodendrocyte",
    "38": "ExcitatoryNeuron",
    "39": "OPC",
    "40": "Bcell",
    "41": "ExcitatoryNeuron",
    "42": "Oligodendrocyte",
    "43": "Oligodendrocyte",
    "44": "Ependymal",
    "45": "ExcitatoryNeuron",
    "46": "Oligodendrocyte",
    "47": "ExcitatoryNeuron",
    "48": "ExcitatoryNeuron",
    "49": "ExcitatoryNeuron",
    "50": "Astrocyte",
    "51": "Microglia",
    "52": "ExcitatoryNeuron",
    "53": "Astrocyte",
    "54": "InhibitoryNeuron",
    "55": "ExcitatoryNeuron",
    "56": "Microglia",
    "57": "Microglia",
    "58": "ExcitatoryNeuron",
    "59": "ExcitatoryNeuron",
    "60": "ExcitatoryNeuron",
    "61": "InhibitoryNeuron",
    "62": "Oligodendrocyte",
    "63": "Astrocyte", 
    "64": "InhibitoryNeuron",
    "65": "COP",
}

leiden_str = macnair_validation_adata.obs["leiden"].astype(str)
mapped = leiden_str.map(leiden_to_celltype)

missing = sorted(leiden_str[mapped.isna()].unique())
if missing:
    raise ValueError(
        f"Missing leiden value in leiden_to_celltype : {missing}\n"
    )
macnair_validation_adata.obs["cell"] = mapped

macnair_validation_adata = macnair_validation_adata[macnair_validation_adata.obs["cell"] != "Doublet"].copy()

In [21]:
macnair_validation_adata.obs["celltype"] = macnair_validation_adata.obs["cell"].copy() #type: ignore
plot.plot_umap(macnair_validation_adata, SERIES, has_celltype=True)

In [ ]:
macnair_validation_adata.write_h5ad(os.path.join(final_h5ad_dir, SERIES + "_final.h5ad"))

oligo= macnair_validation_adata[
    (macnair_validation_adata.obs["cell"] == "OPC") |
    (macnair_validation_adata.obs["cell"] == "COP") | 
    (macnair_validation_adata.obs["cell"] == "Oligodendrocyte")
]
oligo.write_h5ad(os.path.join(pre_h5ad_dir, SERIES + "_oligo_raw.h5ad"))

microglia = macnair_validation_adata[
    (macnair_validation_adata.obs["cell"] == "Microglia")
]
microglia.write_h5ad(os.path.join(pre_h5ad_dir, SERIES + "_microglia_raw.h5ad"))

astrocyte = macnair_validation_adata[
    (macnair_validation_adata.obs["cell"] == "Astrocyte")
]
astrocyte.write_h5ad(os.path.join(pre_h5ad_dir, SERIES + "_astrocyte_raw.h5ad"))

/tmp/ipykernel_2789143/2102708534.py:8: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  oligo.write_h5ad(os.path.join(pre_h5ad_dir, SERIES + "_oligo_raw.h5ad"))
